# Cleaning Data Mentah Pengaduan Surabaya 




In [ ]:
from pathlib import Path
import re, json, unicodedata, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

BASE_DIR = Path.cwd()
SEARCH_DIRS = [BASE_DIR, BASE_DIR / "project_raw", BASE_DIR.parent, BASE_DIR.parent / "project_raw"]

TAHUN_LIST  = [2023, 2024, 2025]
N_KELURAHAN = 153
BBOX_LAT    = (-7.45, -7.10)     # batas pemeriksaan Surabaya
BBOX_LON    = (112.55, 112.90)

KECAMATAN_RESMI = [
    "Asemrowo", "Benowo", "Bubutan", "Bulak", "Dukuh Pakis", "Gayungan", "Genteng", "Gubeng",
    "Gunung Anyar", "Jambangan", "Karang Pilang", "Kenjeran", "Krembangan", "Lakarsantri",
    "Mulyorejo", "Pabean Cantian", "Pakal", "Rungkut", "Sambikerep", "Sawahan", "Semampir",
    "Simokerto", "Sukolilo", "Sukomanunggal", "Tambaksari", "Tandes", "Tegalsari",
    "Tenggilis Mejoyo", "Wiyung", "Wonocolo", "Wonokromo",
]

def find_file(patterns, deskripsi):
    dicari = []
    for d in SEARCH_DIRS:
        for pat in patterns:
            hits = sorted(p for p in d.glob(pat) if not p.name.startswith("~$"))
            if hits:
                return hits[0]
        dicari.append(str(d))
    raise FileNotFoundError(f"File {deskripsi} tidak ditemukan. Folder yang dicari: {dicari}. "
                            f"Pola: {patterns}")

FILES = {
    y: find_file([f"mediacenter.keluhan (Tahun {y}).xlsx", f"*keluhan*{y}*.xlsx"], f"pengaduan {y}")
    for y in TAHUN_LIST
}
COORD_FILE = find_file(
    ["koordinat_kelurahan_2023_2025.csv", "koordinat_kelurahan*2023_2025*.csv", "koordinat_kelurahan*.csv"],
    "koordinat kelurahan",
)


def normalize_location(value):
    if pd.isna(value):
        return pd.NA
    v = unicodedata.normalize("NFKC", str(value)).casefold()
    v = re.sub(r"\s+", " ", v).strip()
    return v if v else pd.NA

def kec_key(value):
    # kunci kecamatan tanpa spasi: 'Gunung Anyar' == 'Gununganyar'
    n = normalize_location(value)
    return pd.NA if pd.isna(n) else n.replace(" ", "")

KEC_RESMI_KEY = {kec_key(k) for k in KECAMATAN_RESMI}
PHONE_RE = re.compile(r"(?<!\d)(?:\+?62|0)[\s.\-]?8\d(?:[\s.\-]?\d){7,10}(?!\d)")

print("Folder kerja :", BASE_DIR)
for y, p in FILES.items():
    print(f"Pengaduan {y}:", p)
print("Koordinat    :", COORD_FILE)

Folder kerja : c:\Users\DELL\Downloads\Bismillah skripsi lancar jaya nilai A
Pengaduan 2023: c:\Users\DELL\Downloads\Bismillah skripsi lancar jaya nilai A\mediacenter.keluhan (Tahun 2023).xlsx
Pengaduan 2024: c:\Users\DELL\Downloads\Bismillah skripsi lancar jaya nilai A\mediacenter.keluhan (Tahun 2024).xlsx
Pengaduan 2025: c:\Users\DELL\Downloads\Bismillah skripsi lancar jaya nilai A\mediacenter.keluhan (Tahun 2025).xlsx
Koordinat    : c:\Users\DELL\Downloads\Bismillah skripsi lancar jaya nilai A\koordinat_kelurahan_2023_2025.csv


In [ ]:
OUTPUT_DIR = BASE_DIR / "cleaning_output"; OUTPUT_DIR.mkdir(exist_ok=True)


KELURAHAN_ALIAS = {}

LOKASI_MANUAL = {}

MASK_TELEPON  = True                
TOKEN_TELEPON = "[NOMOR_TELEPON]"
MASK_PII_LAIN = True                
TOKEN_ID, TOKEN_EMAIL, TOKEN_AKUN = "[NOMOR_IDENTITAS]", "[EMAIL]", "[AKUN]"
MASK_URL  = True                   
TOKEN_URL = "[URL]"

ALLOW_FALLBACK_COORDINATE = False  
SHORT_THRESHOLD = 10                
LONG_THRESHOLD  = 2000             
MIN_N_WARN      = 5                


TOPIK_OVERRIDE = {
    "Parkir Mobil di Jalan Lingkungan/Kawasan Permukiman": "Perhubungan & Lalu Lintas",       # parkir/truk menghalangi akses
    "Pembersihan Sampah Di Saluran/Sungai, Pedestrian, dan Jalan": "Kebersihan & Lingkungan",  # sampah/tanaman di sungai, pasar, taman
    "Marka Jalan": "Perhubungan & Lalu Lintas",            # marka pudar, water barrier (Dinas Perhubungan)
    "Penutupan Jalan Utama": "Perhubungan & Lalu Lintas",  # penutupan/pengalihan jalan
    "Pembatas jalan": "Perhubungan & Lalu Lintas",         # pembatas/JPO/arus lalu lintas
    "CCTV Jalan": "Perhubungan & Lalu Lintas",             # sekitar 15 dari 17 tiket: CCTV lalu lintas / aplikasi SITS
    "Kelas Jalan": "Perhubungan & Lalu Lintas",            # pembatasan truk, portal, tombol penyeberangan
    "Pembersihan Sungai Akibat Sedimen Saluran": "Banjir & Drainase",   # pengerukan sedimen sungai/saluran (sebagian tiket terkait sampah kerja bakti)
    "Normalisasi Saluran Permukiman": "Banjir & Drainase",              # pengerukan selokan/saluran
    "Perbaikan Tanggul / Plengsengan": "Banjir & Drainase",             # tanggul/plengsengan sungai & selokan
    "Pavingisasi": "Infrastruktur Jalan & Saluran",                     # paving/pengaspalan/trotoar
    "Miras (Minuman Keras)": "Keamanan & Ketertiban",      # penertiban miras
    "CCTV Lingkungan": "Keamanan & Ketertiban",            # CCTV RT/RW untuk keamanan (curanmor dll)
    "Beasiswa SMA (Pemuda Tangguh)": "Pendidikan",             # beasiswa = bagian dari pendidikan (konsisten dengan Beasiswa CSR SMP MBR)
    "Beasiswa Mahasiswa": "Pendidikan",
    "Beasiswa Pemuda Tangguh": "Pendidikan",
    "Beasiswa Bibit Unggul": "Pendidikan",
    "Pengerukan Saluran Permukiman": "Banjir & Drainase",      # sedimentasi, pendangkalan selokan, banjir saat hujan (Dinas Sumber Daya Air dan Bina Marga)
    "Saluran Air Di Kampung/Perumahan": "Banjir & Drainase",   # got, U-ditch, plengsengan sungai (Dinas Sumber Daya Air dan Bina Marga)
    "Keamanan Website/Aplikasi Pemerintah Kota Surabaya": "Lainnya",   # isi: aplikasi WargaKu/SSW (Diskominfo), bukan keamanan ketertiban
    "Pajak Penerangan Jalan": "Pajak & Retribusi Daerah",  # substansi pajak (Badan Pendapatan Daerah), bukan penerangan
}


## 1. Load data raw
`Tahun_file` = tahun berdasarkan file sumber (metadata). Tahun analisis (`tahun`) diturunkan dari tanggal di bagian 3.

In [18]:
frames = []
for y, p in FILES.items():
    d = pd.read_excel(p)
    d["Tahun_file"] = y
    frames.append(d)
df_raw = pd.concat(frames, ignore_index=True)
df_raw["raw_row"] = np.arange(len(df_raw))            # penelusuran ke baris asal
print("Shape gabungan:", df_raw.shape)

def _clean_col(c):
    return re.sub(r"\s+", " ", str(c)).strip()

df = df_raw.copy()
df.columns = [_clean_col(c) for c in df.columns]

WAJIB = ["No Ticket", "Instansi", "Topik", "Kategori", "Channel", "Keluhan", "Kelurahan", "Kecamatan", "Tanggal Keluhan"]
hilang = [c for c in WAJIB if c not in df.columns]
assert not hilang, f"Kolom wajib tidak ada: {hilang}"

# Whitespace teknis saja (nilai substantif tidak diubah)
for col in ["Instansi", "Topik", "Kategori", "Channel", "Keluhan", "Kelurahan", "Kecamatan"]:
    df[col] = df[col].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    df[col] = df[col].mask(df[col] == "", pd.NA)
print("Baris mentah:", len(df))

Shape gabungan: (33935, 16)
Baris mentah: 33935


## 2. Tiket & duplikasi

In [19]:
n_dup_baris = int(df.drop(columns=["raw_row"]).duplicated().sum())
n_dup_tiket_dalam = int(df.duplicated(["Tahun_file", "No Ticket"]).sum())
n_dup_tiket_total = int(df["No Ticket"].duplicated().sum())
print("Duplikat baris utuh          :", n_dup_baris)
print("Duplikat tiket dalam 1 tahun :", n_dup_tiket_dalam)
print("Duplikat tiket antar tahun   :", n_dup_tiket_total - n_dup_tiket_dalam)

# Kebijakan: tidak ada dedup otomatis. Jika ada duplikat tiket, berhenti dan putuskan secara sadar.
assert n_dup_tiket_total == 0, "Ada duplikat tiket: tentukan aturan dedup secara eksplisit sebelum lanjut."
print("OK: No Ticket unik pada seluruh data.")

Duplikat baris utuh          : 0
Duplikat tiket dalam 1 tahun : 0
Duplikat tiket antar tahun   : 0
OK: No Ticket unik pada seluruh data.


## 3. Tanggal & tahun analisis
`tahun` diturunkan dari `Tanggal Keluhan` dan **harus** sama dengan tahun file.

In [20]:
df["tanggal_dt"] = pd.to_datetime(df["Tanggal Keluhan"], errors="coerce")
df["tahun"] = df["tanggal_dt"].dt.year

n_tgl_invalid = int(df["tanggal_dt"].isna().sum())
n_tahun_beda  = int((df["tahun"].notna() & (df["tahun"] != df["Tahun_file"])).sum())
print("Tanggal invalid                       :", n_tgl_invalid)
print("Tahun tanggal != tahun file           :", n_tahun_beda)
assert n_tahun_beda == 0, "Tahun pada tanggal tidak sama dengan tahun file: periksa sebelum lanjut."
display(df.groupby("Tahun_file")["tanggal_dt"].agg(["min", "max", "count"]))

# Audit eksplisit: baris dengan tanggal invalid ditampilkan & disimpan (dikeluarkan di bagian 9, jumlahnya dicocokkan di sana)
tgl_invalid = df.loc[df["tanggal_dt"].isna(), ["raw_row", "No Ticket", "Tahun_file", "Tanggal Keluhan", "Kelurahan", "Kecamatan"]]
print(f"\nBaris dengan tanggal invalid ({len(tgl_invalid)}) - tinjau sebelum lanjut:")
display(tgl_invalid.head(20))
tgl_invalid.to_csv(OUTPUT_DIR / "tanggal_invalid_audit.csv", index=False, encoding="utf-8-sig")


Tanggal invalid                       : 0
Tahun tanggal != tahun file           : 0


,min,max,count
Tahun_file,,,
2023,2023-01-01 08:26:00,2023-12-31 23:57:00,12253
2024,2024-01-01 01:03:00,2024-12-31 23:37:00,11495
2025,2025-01-01 12:24:00,2025-12-31 23:48:00,10187



Baris dengan tanggal invalid (0) - tinjau sebelum lanjut:


,raw_row,No Ticket,Tahun_file,Tanggal Keluhan,Kelurahan,Kecamatan


## 4. Master koordinat kelurahan
File koordinat adalah hasil tahap wrangling sebelumnya, jadi kualitasnya divalidasi di sini — bukan diasumsikan benar.

In [21]:
coord = pd.read_csv(COORD_FILE)
need = ["kelurahan", "kecamatan", "lat_kelurahan", "lon_kelurahan"]
assert all(c in coord.columns for c in need), f"Kolom koordinat wajib: {need}"

master = coord.copy().reset_index(drop=True)
master["master_id"] = master.index
master["kelurahan"] = master["kelurahan"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
master["kecamatan"] = master["kecamatan"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
master["kel_norm"]  = master["kelurahan"].map(normalize_location)
master["kec_k"]     = master["kecamatan"].map(kec_key)
master["kec_norm"]  = master["kecamatan"].map(normalize_location)

def _kel_base(kel_norm, kec_norm):
    # Label master yang dibedakan dengan nama kecamatan: 'wonorejo rungkut' (kecamatan 'rungkut') -> 'wonorejo'.
    # Hanya dipakai untuk mendeteksi nama yang ambigu; identitas tetap ditentukan oleh pasangan kelurahan+kecamatan.
    if pd.isna(kel_norm) or pd.isna(kec_norm):
        return kel_norm
    suf = " " + kec_norm
    return kel_norm[:-len(suf)].strip() if kel_norm.endswith(suf) and len(kel_norm) > len(suf) else kel_norm

master["kel_base"]   = [_kel_base(a, b) for a, b in zip(master["kel_norm"], master["kec_norm"])]
master["lokasi_key"] = master["kel_base"].astype(str) + "|" + master["kec_k"].astype(str)   # identitas administratif
master["lat"] = pd.to_numeric(master["lat_kelurahan"], errors="coerce")
master["lon"] = pd.to_numeric(master["lon_kelurahan"], errors="coerce")
if "sumber_koordinat" not in master.columns:
    print("PERINGATAN: kolom sumber_koordinat tidak ada; asal koordinat tidak dapat diaudit.")
    master["sumber_koordinat"] = "tidak_diketahui"

# NB: pengecekan duplikat dilakukan SEBELUM drop_duplicates apa pun.
cek_master = {
    "jumlah_baris": len(master),
    "nama_kelurahan_duplikat": int(master["kel_norm"].duplicated().sum()),
    "identitas_kel_kec_duplikat": int(master["lokasi_key"].duplicated().sum()),
    "label_tanpa_spasi_duplikat": int(master["kel_norm"].str.replace(" ", "", regex=False).duplicated().sum()),
    "koordinat_kosong": int(master[["lat", "lon"]].isna().any(axis=1).sum()),
    "di_luar_bbox": int((~master["lat"].between(*BBOX_LAT) | ~master["lon"].between(*BBOX_LON)).sum()),
    "lat_lon_duplikat": int(master.duplicated(["lat", "lon"]).sum()),
    "kecamatan_non_resmi": int((~master["kec_k"].isin(KEC_RESMI_KEY)).sum()),
    "baris_fallback": int(master["sumber_koordinat"].astype(str).str.contains("fallback", case=False).sum()),
}
display(pd.Series(cek_master, name="nilai").to_frame())
print("sumber_koordinat:"); display(master["sumber_koordinat"].value_counts(dropna=False).rename("n").to_frame())

assert cek_master["jumlah_baris"] == N_KELURAHAN, f"Master harus {N_KELURAHAN} baris."
assert cek_master["identitas_kel_kec_duplikat"] == 0, "Pasangan kelurahan+kecamatan di master tidak unik -> identitas administratif ganda."
assert cek_master["nama_kelurahan_duplikat"] == 0, "Label kelurahan di master tidak unik: bedakan labelnya (mis. tambahkan nama kecamatan) agar tiap kelurahan punya label sendiri."
assert cek_master["label_tanpa_spasi_duplikat"] == 0, "Ada dua label master yang sama bila spasi dihapus."
assert cek_master["koordinat_kosong"] == 0 and cek_master["di_luar_bbox"] == 0, "Koordinat master tidak valid."
assert cek_master["lat_lon_duplikat"] == 0, "Ada koordinat sama dipakai >1 kelurahan."
assert cek_master["kecamatan_non_resmi"] == 0, "Ada kecamatan non-resmi di master."
if not ALLOW_FALLBACK_COORDINATE:
    assert cek_master["baris_fallback"] == 0, "Ada koordinat fallback kecamatan di master (bukan koordinat kelurahan)."
print("OK: master koordinat valid.")

,nilai
jumlah_baris,153
nama_kelurahan_duplikat,0
identitas_kel_kec_duplikat,0
label_tanpa_spasi_duplikat,0
koordinat_kosong,0
di_luar_bbox,0
lat_lon_duplikat,0
kecamatan_non_resmi,0
baris_fallback,0


sumber_koordinat:


,n
sumber_koordinat,
master_csv,153


OK: master koordinat valid.


## 5. Pencocokan lokasi



In [ ]:
df["kel_norm"] = df["Kelurahan"].map(normalize_location)
df["kel_norm"] = df["kel_norm"].map(lambda x: x if pd.isna(x) else KELURAHAN_ALIAS.get(x, x))
df["kec_k_raw"] = df["Kecamatan"].map(kec_key)
df["_kec_key"] = df["kec_k_raw"].fillna("").astype(str)      
LABEL_MAP         = dict(zip(master["kel_norm"], master["master_id"]))
LABEL_KEC_MAP     = {(a, b): i for i, a, b in zip(master["master_id"], master["kel_norm"], master["kec_k"])}
LABEL_TANPA_SPASI = {a.replace(" ", ""): i for i, a in zip(master["master_id"], master["kel_norm"])}
KANDIDAT = {}                                                # nama (label / nama dasar) -> himpunan master_id
for i, lab, base in zip(master["master_id"], master["kel_norm"], master["kel_base"]):
    KANDIDAT.setdefault(lab, set()).add(i)
    KANDIDAT.setdefault(base, set()).add(i)
KELUARGA_NAMA = {b: ids for b, ids in master.groupby("kel_base")["master_id"].apply(list).items() if len(ids) > 1}   # nama kembar
KEC_MASTER  = dict(zip(master["master_id"], master["kec_k"]))
BASE_MASTER = dict(zip(master["master_id"], master["kel_base"]))
for (_k, _c), _lab in LOKASI_MANUAL.items():
    assert _lab in LABEL_MAP, f"LOKASI_MANUAL menunjuk label yang tidak ada di master: {_lab}"

def _resolve(kel, keck):
    if (kel, keck) in LOKASI_MANUAL:
        return LABEL_MAP[LOKASI_MANUAL[(kel, keck)]], "MANUAL_TERVERIFIKASI"
    if (kel, keck) in LABEL_KEC_MAP:
        return LABEL_KEC_MAP[(kel, keck)], "PASANGAN_EXACT"
    if keck and (kel.replace(" ", "") + keck) in LABEL_TANPA_SPASI:
        return LABEL_TANPA_SPASI[kel.replace(" ", "") + keck], "LABEL_MASTER"
    ids = KANDIDAT.get(kel, set())
    if len(ids) == 1:
        i0 = next(iter(ids))
        kembar = KELUARGA_NAMA.get(BASE_MASTER[i0])
        if kembar and keck:                      
            cocok = [j for j in kembar if KEC_MASTER[j] == keck]
            if len(cocok) == 1:
                return cocok[0], "KECAMATAN_PEMBEDA"  
            return None, "AMBIGU"                  
        return i0, "NAMA_UNIK"                   
    if len(ids) > 1:
        return None, "AMBIGU"                     
    return None, "TIDAK_ADA"

pairs = df.loc[df["kel_norm"].notna(), ["kel_norm", "_kec_key"]].drop_duplicates().reset_index(drop=True)
_hasil = [_resolve(a, b) for a, b in zip(pairs["kel_norm"], pairs["_kec_key"])]
pairs["master_id"] = pd.array([h[0] for h in _hasil], dtype="Int64")
pairs["metode_cocok"] = [h[1] for h in _hasil]

df = df.drop(columns=[c for c in ["master_id", "metode_cocok", "kelurahan_master", "kecamatan_master", "kec_k_master",
                                  "lat", "lon", "sumber_koordinat", "lokasi_key", "kode_kel"] if c in df.columns])
df = df.merge(pairs, on=["kel_norm", "_kec_key"], how="left", validate="many_to_one")
df["metode_cocok"] = df["metode_cocok"].fillna("MISSING_LOCATION")

m_cols = ["master_id", "kelurahan", "kecamatan", "kec_k", "lat", "lon", "sumber_koordinat", "lokasi_key"]
if "kode_kel" in master.columns:                            
    m_cols.append("kode_kel")
m = master[m_cols].rename(columns={"kelurahan": "kelurahan_master", "kecamatan": "kecamatan_master", "kec_k": "kec_k_master"})
m["master_id"] = m["master_id"].astype("Int64")
df = df.merge(m, on="master_id", how="left", validate="many_to_one")

df["coordinate_status"] = np.select(
    [df["kel_norm"].isna(),
     df["metode_cocok"] == "AMBIGU",
     df["lat"].isna() | df["lon"].isna(),
     ~df["lat"].between(*BBOX_LAT) | ~df["lon"].between(*BBOX_LON)],
    ["MISSING_LOCATION", "AMBIGUOUS_LOCATION", "NO_COORDINATE_MATCH", "INVALID_COORDINATE"],
    default="MATCH_COORDINATE",
)
display(pd.crosstab(df["Tahun_file"], df["coordinate_status"], margins=True))
print("Metode pencocokan:")
display(df["metode_cocok"].value_counts().rename("n_baris").to_frame())

tak_cocok = (df.loc[df["coordinate_status"] == "NO_COORDINATE_MATCH", ["Kelurahan", "Kecamatan"]]
             .value_counts(dropna=False).reset_index(name="n_baris"))
print("Nama kelurahan di pengaduan yang TIDAK ada di master koordinat:")
display(tak_cocok)
if len(tak_cocok):
    print("-> Perbaiki lewat KELURAHAN_ALIAS di sel konfigurasi (mis. beda ejaan), lalu jalankan ulang.")
tak_cocok.to_csv(OUTPUT_DIR / "lokasi_tidak_cocok_audit.csv", index=False, encoding="utf-8-sig")

ambigu = (df.loc[df["coordinate_status"] == "AMBIGUOUS_LOCATION", ["Kelurahan", "Kecamatan"]]
          .value_counts(dropna=False).reset_index(name="n_baris"))
print("Nama kelurahan AMBIGU (lebih dari satu kandidat di master; kecamatan di data tidak membedakan):")
display(ambigu)
if len(ambigu):
    print("-> Jangan dikoreksi dari nama saja. Verifikasi ke sumber administratif, lalu isi LOKASI_MANUAL, dan jalankan ulang.")
ambigu.to_csv(OUTPUT_DIR / "lokasi_ambigu_audit.csv", index=False, encoding="utf-8-sig")


coordinate_status,MATCH_COORDINATE,MISSING_LOCATION,All
Tahun_file,,,
2023,12221,32,12253
2024,11483,12,11495
2025,10180,7,10187
All,33884,51,33935


Metode pencocokan:


,n_baris
metode_cocok,
PASANGAN_EXACT,33865
MISSING_LOCATION,51
KECAMATAN_PEMBEDA,18
NAMA_UNIK,1


Nama kelurahan di pengaduan yang TIDAK ada di master koordinat:


,Kelurahan,Kecamatan,n_baris


Nama kelurahan AMBIGU (lebih dari satu kandidat di master; kecamatan di data tidak membedakan):


,Kelurahan,Kecamatan,n_baris


In [ ]:
ada = df["master_id"].notna()
beda = df["kec_k_raw"].fillna("").astype(str) != df["kec_k_master"].fillna("").astype(str)
df["kecamatan_dikoreksi"] = (ada & beda).astype(bool)

audit_kec = (df.loc[df["kecamatan_dikoreksi"], ["Kelurahan", "Kecamatan", "kelurahan_master", "kecamatan_master", "metode_cocok"]]
             .value_counts(dropna=False).reset_index(name="n_baris")
             .rename(columns={"Kecamatan": "kecamatan_di_data", "kecamatan_master": "kecamatan_dikoreksi_ke"}))
print("Baris dengan Kecamatan salah-input yang dikoreksi mengikuti master:", int(df["kecamatan_dikoreksi"].sum()))
display(audit_kec)
audit_kec.to_csv(OUTPUT_DIR / "kecamatan_correction_audit.csv", index=False, encoding="utf-8-sig")


df["kelurahan_dikoreksi"] = (df["metode_cocok"] == "KECAMATAN_PEMBEDA").astype(bool)
audit_kel = df.loc[df["kelurahan_dikoreksi"], ["No Ticket", "Tanggal Keluhan", "Kelurahan", "Kecamatan", "kelurahan_master", "kecamatan_master", "Topik", "Keluhan"]].copy()
audit_kel["Keluhan"] = audit_kel["Keluhan"].astype(str).str.replace(r"\s+", " ", regex=True).str.slice(0, 160)
audit_kel = audit_kel.rename(columns={"Kelurahan": "kelurahan_di_data", "Kecamatan": "kecamatan_di_data", "kelurahan_master": "kelurahan_dikoreksi_ke"})
print("Baris dengan LABEL kelurahan (nama kembar) dikoreksi mengikuti kecamatan mentah:", int(df["kelurahan_dikoreksi"].sum()))
display(audit_kel.drop(columns=["Keluhan", "Tanggal Keluhan"]).groupby(["kelurahan_di_data", "kecamatan_di_data", "kelurahan_dikoreksi_ke"]).size().rename("n_baris").reset_index())
audit_kel.to_csv(OUTPUT_DIR / "kelurahan_label_correction_audit.csv", index=False, encoding="utf-8-sig")


Baris dengan Kecamatan salah-input yang dikoreksi mengikuti master: 1


,Kelurahan,kecamatan_di_data,kelurahan_master,kecamatan_dikoreksi_ke,metode_cocok,n_baris
0,Jemur Wonosari,Tenggilis Mejoyo,Jemur Wonosari,Wonocolo,NAMA_UNIK,1


Baris dengan LABEL kelurahan (nama kembar) dikoreksi mengikuti kecamatan mentah: 18


,kelurahan_di_data,kecamatan_di_data,kelurahan_dikoreksi_ke,n_baris
0,Wonorejo Rungkut,Tegalsari,Wonorejo Tegalsari,7
1,Wonorejo Tegalsari,Rungkut,Wonorejo Rungkut,11


## 6. Audit missing Kelurahan
Missing **tidak diimputasi**. Karena polanya tidak acak antar kecamatan, dampaknya dihitung per kecamatan.

In [24]:
m_kel = df["kel_norm"].isna()
print("Missing Kelurahan:", int(m_kel.sum()), f"({m_kel.mean()*100:.2f}%)")
print("Missing Kelurahan tetapi Kecamatan ada:", int((m_kel & df["Kecamatan"].notna()).sum()))

dampak = (df.groupby("Kecamatan").agg(total=("raw_row", "size"), missing_kelurahan=("kel_norm", lambda s: int(s.isna().sum())))
          .assign(persen_missing=lambda t: (t["missing_kelurahan"] / t["total"] * 100).round(2))
          .sort_values("missing_kelurahan", ascending=False))
print("Dampak per Kecamatan (10 teratas):")
display(dampak.head(10))
dampak.to_csv(OUTPUT_DIR / "missing_kelurahan_impact_per_kecamatan.csv", encoding="utf-8-sig")
display(pd.crosstab(df.loc[m_kel, "Kecamatan"], df.loc[m_kel, "Tahun_file"], margins=True).sort_values("All", ascending=False).head(8))

Missing Kelurahan: 51 (0.15%)
Missing Kelurahan tetapi Kecamatan ada: 51
Dampak per Kecamatan (10 teratas):


,total,missing_kelurahan,persen_missing
Kecamatan,,,
Pabean Cantian,494,40,8.10
Asemrowo,351,7,1.99
Bubutan,816,3,0.37
Tandes,943,1,0.11
Dukuh Pakis,1014,0,0.00
Bulak,309,0,0.00
Benowo,797,0,0.00
Gubeng,3424,0,0.00
Gununganyar,687,0,0.00


Tahun_file,2023,2024,2025,All
Kecamatan,,,,
All,32,12,7,51
Pabean Cantian,21,12,7,40
Asemrowo,7,0,0,7
Bubutan,3,0,0,3
Tandes,1,0,0,1


## 7. Teks `Keluhan`: cleaning ringan, audit ekstrem, duplikat, data pribadi


In [ ]:
def clean_complaint_text(v):
    if pd.isna(v):
        return pd.NA
    v = str(v)
    
    v = re.sub(r"\\[nrt]", " ", v)
    v = unicodedata.normalize("NFKC", v)
    v = re.sub(r"\s+", " ", v).strip()
    return v if v else pd.NA

df["Keluhan_clean"] = df["Keluhan"].map(clean_complaint_text)
m_keluhan = df["Keluhan_clean"].isna()
print("Missing/kosong Keluhan:", int(m_keluhan.sum()))

_raw_txt = df["Keluhan"].astype("string")
n_artefak_raw = int(_raw_txt.str.contains(r"\\[nrt]", regex=True, na=False).sum())
n_artefak_sisa = int(df["Keluhan_clean"].astype("string").str.contains(r"\\[nrt]", regex=True, na=False).sum())
print(f"Baris dgn literal \\\\n/\\\\r/\\\\t di teks mentah : {n_artefak_raw}")
print(f"Sisa setelah cleaning (harus 0)           : {n_artefak_sisa}")
assert n_artefak_sisa == 0, "Masih ada literal backslash-escape tersisa setelah cleaning."

s = df["Keluhan_clean"].astype("string")
df["keluhan_char_count"] = s.str.len().fillna(0).astype(int)
df["keluhan_word_count"] = s.str.split().str.len().fillna(0).astype(int)
display(df.groupby("Tahun_file")[["keluhan_char_count", "keluhan_word_count"]].describe().T.round(1))


pendek = df[(df["keluhan_char_count"] > 0) & (df["keluhan_char_count"] <= SHORT_THRESHOLD)]
panjang = df[df["keluhan_char_count"] >= LONG_THRESHOLD]
print(f"\nTeks pendek (1..{SHORT_THRESHOLD} karakter): {len(pendek)} | teks panjang (>={LONG_THRESHOLD}): {len(panjang)}")
print("Isi teks pendek yang paling sering (apakah masih bermakna?):")
display(pendek["Keluhan_clean"].astype("string").str.lower().value_counts().head(12).rename("n").to_frame())
pd.concat([pendek.assign(extreme_type="VERY_SHORT_AUDIT"), panjang.assign(extreme_type="VERY_LONG_AUDIT")]) \
  [["No Ticket", "Tahun_file", "Kelurahan", "Keluhan_clean", "keluhan_char_count", "extreme_type"]] \
  .to_csv(OUTPUT_DIR / "text_extreme_audit.csv", index=False, encoding="utf-8-sig")

key_t = df["Keluhan_clean"].astype("string").str.casefold()
valid_key = key_t.notna() & df["kel_norm"].notna()
tmp = pd.DataFrame({"t": key_t, "k": df["kel_norm"].astype("string"), "y": df["tahun"]})
df["flag_dup_teks"] = valid_key & tmp.duplicated(keep=False)
print("\nBaris dengan teks identik di kelurahan & tahun yang sama (flag_dup_teks):", int(df["flag_dup_teks"].sum()))
rep = (tmp[df["flag_dup_teks"]].groupby(["t", "k", "y"]).size().reset_index(name="n").sort_values("n", ascending=False).head(8))
rep["t"] = rep["t"].str.slice(0, 80)
display(rep)

df["flag_telepon"] = s.fillna("").str.contains(PHONE_RE)
print("Baris memuat pola nomor telepon (flag_telepon):", int(df["flag_telepon"].sum()))


EMAIL_RE = re.compile(r"[\w.\-+]+@[\w\-]+(?:\.[\w\-]+)+")
ID_RE    = re.compile(r"(?<!\d)\d{15,}(?!\d)")            # NIK / No. KK / nomor identitas panjang lain
AKUN_RE  = re.compile(r"(?<![\w@])@\w{3,}")                 # @akun media sosial
URL_RE   = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)

def mask_telepon(v):
    if pd.isna(v):
        return pd.NA
    v = str(v)
    if MASK_PII_LAIN:                                         # urutan penting: url -> email -> identitas panjang -> telepon -> akun
        if MASK_URL:
            v = URL_RE.sub(TOKEN_URL, v)
        v = EMAIL_RE.sub(TOKEN_EMAIL, v)
        v = ID_RE.sub(TOKEN_ID, v)
    v = PHONE_RE.sub(TOKEN_TELEPON, v)
    if MASK_PII_LAIN:
        v = AKUN_RE.sub(TOKEN_AKUN, v)
    return v

df["Keluhan_embedding"] = df["Keluhan_clean"].map(mask_telepon) if MASK_TELEPON else df["Keluhan_clean"]
n_dimasking = int((df["Keluhan_embedding"].astype("string") != df["Keluhan_clean"].astype("string")).fillna(False).sum())
print(f"Baris yang teks_embedding-nya dimasking (MASK_TELEPON={MASK_TELEPON}, MASK_PII_LAIN={MASK_PII_LAIN}):", n_dimasking)
for _nm, _tok in [("telepon", TOKEN_TELEPON), ("identitas panjang", TOKEN_ID), ("email", TOKEN_EMAIL),
                  ("akun", TOKEN_AKUN), ("url", TOKEN_URL)]:
    print(f"  - memuat {_tok} ({_nm}):", int(df["Keluhan_embedding"].astype("string").str.contains(_tok, regex=False).fillna(False).sum()))

_emb = df["Keluhan_embedding"].astype("string").fillna("")
assert not _emb.str.contains(PHONE_RE).any(), "Masih ada pola telepon di teks_embedding"
assert not _emb.str.contains(EMAIL_RE).any(), "Masih ada email di teks_embedding"
AUDIT_RE = re.compile(r"(?:\d[\s.\-]?){9,}")                  # deret >= 9 digit: bisa nomor pelanggan/NOP/KTP
_aud = _emb[_emb.str.contains(AUDIT_RE)]
print(f"\nAudit longgar: {len(_aud)} baris masih memuat deret >= 9 digit (perlu ditinjau manual; bisa nomor pelanggan/objek pajak/dokumen).")
display(_aud.str.extract("(" + AUDIT_RE.pattern + ")")[0].str.replace(r"\d", "#", regex=True).str.strip()
        .value_counts().head(8).rename("n").to_frame())


Missing/kosong Keluhan: 4
Baris dgn literal \\n/\\r/\\t di teks mentah : 8596
Sisa setelah cleaning (harus 0)           : 0


Tahun_file                   2023     2024     2025
keluhan_char_count count  12253.0  11495.0  10187.0
                   mean     306.8    290.5    299.5
                   std      388.3    389.8    500.6
                   min        0.0      0.0      4.0
                   25%      100.0     93.0     88.0
                   50%      192.0    178.0    176.0
                   75%      366.0    338.0    335.0
                   max    10700.0   7414.0  15837.0
keluhan_word_count count  12253.0  11495.0  10187.0
                   mean      46.8     44.3     45.6
                   std       59.0     59.3     77.3
                   min        0.0      0.0      1.0
                   25%       15.0     14.5     14.0
                   50%       29.0     27.0     27.0
                   75%       56.0     51.5     50.0
                   max     1680.0   1305.0   2265.0


Teks pendek (1..10 karakter): 125 | teks panjang (>=2000): 293
Isi teks pendek yang paling sering (apakah masih bermakna?):


,n
Keluhan_clean,
pju mati,15
pju padam,14
jln lubang,12
lubang,12
jl rusak,9
jln rsk,5
jln rusak,4
jln ambles,3
jl lubang,3



Baris dengan teks identik di kelurahan & tahun yang sama (flag_dup_teks): 552


,t,k,y,n
196,"selamat pagi, perkenalkan nama saya rendi. saya akan melaporkan oknum pegawai di",kebonsari,2025,21
43,jalan berlubang,ngagel,2025,8
165,retak dan lubang,alun-alun contong,2025,6
55,jalan rusak,jemur wonosari,2025,5
83,jln praban rak semua lubang plus retak,alun-alun contong,2025,5
44,jalan berlubang,perak barat,2023,5
56,jalan rusak,sidosermo,2025,5
66,jalan tidak rata,kertajaya,2025,5


Baris memuat pola nomor telepon (flag_telepon): 1263
Baris yang teks_embedding-nya dimasking (MASK_TELEPON=True, MASK_PII_LAIN=True): 1784
  - memuat [NOMOR_TELEPON] (telepon): 1263
  - memuat [NOMOR_IDENTITAS] (identitas panjang): 433
  - memuat [EMAIL] (email): 84
  - memuat [AKUN] (akun): 47
  - memuat [URL] (url): 136

Audit longgar: 201 baris masih memuat deret >= 9 digit (perlu ditinjau manual; bisa nomor pelanggan/objek pajak/dokumen).


,n
0,
##########,33
##.##.####.##.####,19
###########,18
#############,16
#########,7
##-##-#### ##,7
##-##-###-###-###-####-#,6
##############,6


## 8. Kelompok topik (16 kelompok)

In [ ]:
def kelompok_topik(topik, instansi):
    teks = str(topik).lower() + " " + str(instansi).lower()
    if any(k in teks for k in ["pju", "lampu jalan", "penerangan jalan", "traffic light", "lampu penerangan"]):
        return "Penerangan Jalan & Traffic Light"
    if any(k in teks for k in ["jalan rusak", "jalan berlubang", "pengaspalan", "trotoar", "pedestrian", "jalan", "saluran tepi jalan"]):
        return "Infrastruktur Jalan & Saluran"
    if any(k in teks for k in ["banjir", "genangan", "drainase"]):
        return "Banjir & Drainase"
    if any(k in teks for k in ["sampah", "pemangkasan", "perantingan", "pohon", "kebersihan", "lingkungan"]):
        return "Kebersihan & Lingkungan"
    if any(k in teks for k in ["parkir", "rambu", "zebra cross", "bollard", "angkutan", "dishub", "lalu lintas", "perhubungan"]):
        return "Perhubungan & Lalu Lintas"
    if any(k in teks for k in ["kesehatan", "puskesmas", "bpjs", "rumah sakit", "catin"]):
        return "Kesehatan"
    if any(k in teks for k in ["ktp", "kk", "akta", "kependudukan", "dukcapil", "catatan sipil"]):
        return "Kependudukan & Catatan Sipil"
    if any(k in teks for k in ["pendidikan", "sekolah", "ijazah", "spp", "pip", "kip"]):
        return "Pendidikan"
    if any(k in teks for k in ["keamanan", "ketertiban", "preman", "balap liar", "satpol pp", "satpol"]):
        return "Keamanan & Ketertiban"
    if any(k in teks for k in ["pbb", "bphtb", "pajak", "retribusi"]):
        return "Pajak & Retribusi Daerah"
    if any(k in teks for k in ["perumahan", "pertanahan", "pbg", "skrk", "rusunawa", "rumah susun"]):
        return "Perumahan & Pertanahan"
    if any(k in teks for k in ["sosial", "kemiskinan", "miskin", "mbr", "bansos"]):
        return "Sosial"
    if any(k in teks for k in ["tenaga kerja", "lowongan", "pemagangan", "disnaker"]):
        return "Ketenagakerjaan"
    if any(k in teks for k in ["pdam", "air bersih", "kabel", "utilitas", "listrik"]):
        return "Utilitas"
    if any(k in teks for k in ["kecamatan", "kelurahan", "kewilayahan", "pelayanan kelurahan"]):
        return "Pelayanan Kewilayahan"
    return "Lainnya"

KELOMPOK_VALID = {
    "Penerangan Jalan & Traffic Light", "Infrastruktur Jalan & Saluran", "Banjir & Drainase",
    "Kebersihan & Lingkungan", "Perhubungan & Lalu Lintas", "Kesehatan", "Kependudukan & Catatan Sipil",
    "Pendidikan", "Keamanan & Ketertiban", "Pajak & Retribusi Daerah", "Perumahan & Pertanahan",
    "Sosial", "Ketenagakerjaan", "Utilitas", "Pelayanan Kewilayahan", "Lainnya",
}
assert set(TOPIK_OVERRIDE.values()) <= KELOMPOK_VALID, "TOPIK_OVERRIDE memuat nama kelompok yang tidak dikenal."

def _kelompok_final(t, i):
    if (t, i) in TOPIK_OVERRIDE:
        return TOPIK_OVERRIDE[(t, i)]
    if t in TOPIK_OVERRIDE:
        return TOPIK_OVERRIDE[t]
    return kelompok_topik(t, i)

_topik_ada = set(df["Topik"].dropna()); _pair_ada = set(zip(df["Topik"], df["Instansi"]))
_tak_terpakai = [k for k in TOPIK_OVERRIDE if (k not in _pair_ada if isinstance(k, tuple) else k not in _topik_ada)]
assert not _tak_terpakai, f"TOPIK_OVERRIDE memuat kunci yang tidak ada di data (salah ketik?): {_tak_terpakai}"

kombinasi = df[["Topik", "Instansi"]].drop_duplicates().copy()
kombinasi["kelompok_topik"] = [
    _kelompok_final(t, i) for t, i in zip(kombinasi["Topik"], kombinasi["Instansi"])
]
df = df.drop(columns=["kelompok_topik"], errors="ignore").merge(kombinasi, on=["Topik", "Instansi"], how="left")
assert df["kelompok_topik"].notna().all() and set(df["kelompok_topik"]) <= KELOMPOK_VALID

dist = (df["kelompok_topik"].value_counts() / len(df) * 100).round(1).rename("persen").to_frame()
display(dist)

print("\nAudit: topik yang jatuh ke 'Lainnya' (kandidat salah kelompok):")
display(df.loc[df["kelompok_topik"] == "Lainnya", "Topik"].value_counts().head(12).rename("n").to_frame())
print("Audit: topik terbanyak di 'Pelayanan Kewilayahan' (kelompok penampung; periksa apakah ada topik infrastruktur/drainase):")
display(df.loc[df["kelompok_topik"] == "Pelayanan Kewilayahan", "Topik"].value_counts().head(12).rename("n").to_frame())

n_komb = df.groupby(["Topik", "Instansi"], dropna=False).size().rename("n_tiket").reset_index()
komb_n = kombinasi.merge(n_komb, on=["Topik", "Instansi"], how="left")

sampel = komb_n.sample(frac=1, random_state=42).groupby("kelompok_topik").head(5).sort_values(["kelompok_topik", "n_tiket"], ascending=[True, False])
sampel.to_csv(OUTPUT_DIR / "topik_sample_audit.csv", index=False, encoding="utf-8-sig")
print("\nSampel kombinasi Topik+Instansi per kelompok disimpan ke topik_sample_audit.csv (periksa apakah kelompoknya tepat).")

KATA_LUAS = ["jalan", "kk", "pip", "kip", "spp", "mbr", "pju"]
teks_k = komb_n["Topik"].astype("string").fillna("").str.lower() + " " + komb_n["Instansi"].astype("string").fillna("").str.lower()
luas = pd.concat([komb_n[teks_k.str.contains(k, regex=False)].assign(kata_luas=k) for k in KATA_LUAS], ignore_index=True)
luas = luas.sort_values("n_tiket", ascending=False)
luas.to_csv(OUTPUT_DIR / "topik_kata_luas_audit.csv", index=False, encoding="utf-8-sig")
print("Audit: kombinasi yang mengandung kata kunci luas (mis. 'pip' juga menangkap 'pipa'; 'jalan' juga 'perjalanan'):")
display(luas.head(25))

komb_n["kelompok_aturan"] = [kelompok_topik(t, i) for t, i in zip(komb_n["Topik"], komb_n["Instansi"])]
efek = (komb_n[komb_n["kelompok_aturan"] != komb_n["kelompok_topik"]]
        .groupby(["Topik", "kelompok_aturan", "kelompok_topik"])["n_tiket"].sum().reset_index()
        .sort_values("n_tiket", ascending=False))
efek.to_csv(OUTPUT_DIR / "topik_override_effect.csv", index=False, encoding="utf-8-sig")
print(f"\nEfek TOPIK_OVERRIDE: {int(efek['n_tiket'].sum())} tiket berpindah kelompok")
display(efek)


,persen
kelompok_topik,
Kebersihan & Lingkungan,17.3
Infrastruktur Jalan & Saluran,12.4
Pelayanan Kewilayahan,12.2
Penerangan Jalan & Traffic Light,9.2
Perhubungan & Lalu Lintas,9.1
Lainnya,6.4
Kependudukan & Catatan Sipil,5.4
Kesehatan,5.2
Utilitas,5.1



Audit: topik yang jatuh ke 'Lainnya' (kandidat salah kelompok):


,n
Topik,
PKL,177
Pelayanan Perizinan IPT,143
Website sswalfa,143
Aplikasi WargaKu,126
Pelayanan SSW,110
Puspaga,104
Bantuan UMKM (BPUM),91
Pengecekan Disposisi Surat Masuk kepada Walikota,70
Aset Pemerintah Kota Surabaya,54


Audit: topik terbanyak di 'Pelayanan Kewilayahan' (kelompok penampung; periksa apakah ada topik infrastruktur/drainase):


,n
Topik,
Permasalahan Kewilayahan Kelurahan,827
Permasalahan Kewilayahan Kecamatan,734
Outreach,332
Pelayanan Surat Pengantar Nikah,313
Pelayanan Petugas Kelurahan,245
Perangkat RT/RW,239
Pekerjaan Dana Kelurahan (Dakel),153
Layanan Loket Kelurahan,148
"Pelayanan Surat Keterangan Domisili (Lembaga Berbadan Hukum, Lembaga Berbadan Usaha, Lembaga Non Berbadan Hukum dan Lembaga Non Berbadan Usaha)",128



Sampel kombinasi Topik+Instansi per kelompok disimpan ke topik_sample_audit.csv (periksa apakah kelompoknya tepat).
Audit: kombinasi yang mengandung kata kunci luas (mis. 'pip' juga menangkap 'pipa'; 'jalan' juga 'perjalanan'):


,Topik,Instansi,kelompok_topik,n_tiket,kata_luas
1,Jalan Rusak dan Berlubang,Dinas Sumber Daya Air dan Bina Marga,Infrastruktur Jalan & Saluran,2742,jalan
50,PJU,Dinas Perhubungan,Penerangan Jalan & Traffic Light,2740,pju
2,"Pembersihan Sampah Di Saluran/Sungai, Pedestrian, dan Jalan",Dinas Lingkungan Hidup,Kebersihan & Lingkungan,983,jalan
0,Saluran Tepi Jalan,Dinas Sumber Daya Air dan Bina Marga,Infrastruktur Jalan & Saluran,599,jalan
44,KK (Kartu Keluarga),Dinas Kependudukan dan Pencatatan Sipil,Kependudukan & Catatan Sipil,264,kk
4,Pengaspalan jalan,Dinas Sumber Daya Air dan Bina Marga,Infrastruktur Jalan & Saluran,222,jalan
46,Informasi Program Indonesia Pintar (PIP/KIP) SD,Dinas Pendidikan,Pendidikan,167,kip
45,Informasi Program Indonesia Pintar (PIP/KIP) SD,Dinas Pendidikan,Pendidikan,167,pip
3,Marka Jalan,Dinas Perhubungan,Perhubungan & Lalu Lintas,125,jalan
47,Aduan Tunggakan SPP,Dinas Pendidikan,Pendidikan,80,spp



Efek TOPIK_OVERRIDE: 2676 tiket berpindah kelompok


,Topik,kelompok_aturan,kelompok_topik,n_tiket
16,"Pembersihan Sampah Di Saluran/Sungai, Pedestrian, dan Jalan",Infrastruktur Jalan & Saluran,Kebersihan & Lingkungan,983
11,Normalisasi Saluran Permukiman,Pelayanan Kewilayahan,Banjir & Drainase,273
14,Pavingisasi,Lainnya,Infrastruktur Jalan & Saluran,250
17,Pembersihan Sungai Akibat Sedimen Saluran,Lainnya,Banjir & Drainase,212
21,Saluran Air Di Kampung/Perumahan,Perumahan & Pertanahan,Banjir & Drainase,181
13,Parkir Mobil di Jalan Lingkungan/Kawasan Permukiman,Infrastruktur Jalan & Saluran,Perhubungan & Lalu Lintas,166
8,Marka Jalan,Infrastruktur Jalan & Saluran,Perhubungan & Lalu Lintas,125
20,Perbaikan Tanggul / Plengsengan,Lainnya,Banjir & Drainase,115
3,Beasiswa SMA (Pemuda Tangguh),Lainnya,Pendidikan,71
18,Pengerukan Saluran Permukiman,Lainnya,Banjir & Drainase,52


## 9. Keputusan inklusi


In [ ]:
m_tgl = df["tanggal_dt"].isna()
alasan = pd.DataFrame({
    "MISSING_KELURAHAN": df["kel_norm"].isna(),
    "MISSING_KELUHAN": df["Keluhan_clean"].isna(),
    "INVALID_DATE": m_tgl,
})
assert int(alasan["INVALID_DATE"].sum()) == n_tgl_invalid, "Jumlah tanggal invalid di bagian 3 tidak sama dengan yang dikeluarkan."
df["exclusion_reason"] = alasan.apply(lambda r: ";".join(r.index[r]), axis=1)
df["include_in_clean"] = df["exclusion_reason"] == ""

df_clean = df[df["include_in_clean"]].copy()
excluded = df[~df["include_in_clean"]].copy()

print("Raw:", len(df), "| Clean:", len(df_clean), "| Excluded:", len(excluded))
display(df["exclusion_reason"].replace("", "(dipertahankan)").value_counts().rename("n").to_frame())
excluded[["raw_row", "No Ticket", "Tahun_file", "Kelurahan", "Kecamatan", "Topik", "Keluhan", "exclusion_reason"]] \
    .to_csv(OUTPUT_DIR / "excluded_rows.csv", index=False, encoding="utf-8-sig")

n_multi = int(excluded["exclusion_reason"].str.contains(";", regex=False).sum())
print(f"Baris unik dikeluarkan: {len(excluded)} | di antaranya punya >1 alasan: {n_multi}")
print("Kemunculan tiap alasan (bisa tumpang tindih, total boleh melebihi jumlah baris unik):")
display(alasan.sum().astype(int).rename("n_baris_dengan_alasan").to_frame())


Raw: 33935 | Clean: 33880 | Excluded: 55


,n
exclusion_reason,
(dipertahankan),33880
MISSING_KELURAHAN,51
MISSING_KELUHAN,4


Baris unik dikeluarkan: 55 | di antaranya punya >1 alasan: 0
Kemunculan tiap alasan (bisa tumpang tindih, total boleh melebihi jumlah baris unik):


,n_baris_dengan_alasan
MISSING_KELURAHAN,51
MISSING_KELUHAN,4
INVALID_DATE,0


## 10. Validasi (hard checks)
Semua pemeriksaan harus **PASS** sebelum file clean ditulis.

In [28]:
per_th = df_clean.groupby("tahun")["lokasi_key"].nunique()
per_id_kec = df_clean.groupby("lokasi_key")["kecamatan_master"].nunique()
per_id_lat = df_clean.groupby("lokasi_key")[["lat", "lon"]].nunique()
per_id_lbl = df_clean.groupby("lokasi_key")["kelurahan_master"].nunique()
pair_ke_id = df_clean.groupby(["kelurahan_master", "kecamatan_master"])["lokasi_key"].nunique()
ID_MASTER  = set(master["lokasi_key"])
wakil_thn  = df_clean.groupby("tahun")["lokasi_key"].agg(lambda s: set(s) == ID_MASTER)

checks = {
    "raw = clean + excluded": len(df) == len(df_clean) + len(excluded),
    "tiket unik pada data clean": df_clean["No Ticket"].is_unique,
    "tidak ada Kelurahan kosong": df_clean["kel_norm"].notna().all(),
    "tidak ada Keluhan kosong": df_clean["Keluhan_clean"].notna().all(),
    "tidak ada tanggal invalid": df_clean["tanggal_dt"].notna().all(),
    "tahun (tanggal) == tahun (file)": (df_clean["tahun"] == df_clean["Tahun_file"]).all(),
    "semua baris clean MATCH_COORDINATE": (df_clean["coordinate_status"] == "MATCH_COORDINATE").all(),
    "tidak ada lokasi AMBIGU pada data clean": (df_clean["metode_cocok"] != "AMBIGU").all(),
    "koreksi kecamatan hanya lewat identitas tak ambigu": df_clean.loc[df_clean["kecamatan_dikoreksi"], "metode_cocok"].isin(["NAMA_UNIK", "LABEL_MASTER", "MANUAL_TERVERIFIKASI"]).all(),
    "koreksi label kelurahan hanya untuk nama kembar & kecamatan mentah = kecamatan master": bool(
        df_clean.loc[df_clean["kelurahan_dikoreksi"], "kec_k_raw"].astype(str).eq(df_clean.loc[df_clean["kelurahan_dikoreksi"], "kec_k_master"].astype(str)).all()
        and df_clean.loc[df_clean["kelurahan_dikoreksi"], "lokasi_key"].str.split("|").str[0].isin(KELUARGA_NAMA.keys()).all()),
    "tanpa koordinat fallback": ALLOW_FALLBACK_COORDINATE or (~df_clean["sumber_koordinat"].astype(str).str.contains("fallback", case=False)).all(),
    f"tepat {N_KELURAHAN} identitas (kelurahan+kecamatan) pada setiap tahun": bool((per_th == N_KELURAHAN).all() and len(per_th) == len(TAHUN_LIST)),
    f"seluruh {N_KELURAHAN} identitas administratif master terwakili pada setiap tahun": bool(wakil_thn.all() and len(wakil_thn) == len(TAHUN_LIST)),
    "tiap identitas tepat 1 kecamatan": bool((per_id_kec == 1).all()),
    "tiap identitas tepat 1 koordinat": bool((per_id_lat == 1).all().all()),
    "tiap identitas tepat 1 label kelurahan": bool((per_id_lbl == 1).all()),
    "tiap pasangan (kelurahan, kecamatan) tepat 1 identitas": bool((pair_ke_id == 1).all()),
    "kelompok_topik valid": set(df_clean["kelompok_topik"]) <= KELOMPOK_VALID,
}
hasil = pd.DataFrame({"pemeriksaan": list(checks), "status": ["PASS" if v else "FAIL" for v in checks.values()]})
display(hasil)
gagal = hasil[hasil["status"] == "FAIL"]["pemeriksaan"].tolist()
assert not gagal, f"Validasi GAGAL: {gagal}"
print("SEMUA VALIDASI PASS.")


,pemeriksaan,status
0,raw = clean + excluded,PASS
1,tiket unik pada data clean,PASS
2,tidak ada Kelurahan kosong,PASS
3,tidak ada Keluhan kosong,PASS
4,tidak ada tanggal invalid,PASS
5,tahun (tanggal) == tahun (file),PASS
6,semua baris clean MATCH_COORDINATE,PASS
7,tidak ada lokasi AMBIGU pada data clean,PASS
8,koreksi kecamatan hanya lewat identitas tak ambigu,PASS
9,koreksi label kelurahan hanya untuk nama kembar & kecamatan mentah = kecamatan master,PASS


SEMUA VALIDASI PASS.


## 11. Audit coverage

In [ ]:
ringkas_tahun = (df.groupby("Tahun_file").agg(raw=("raw_row", "size"))
                 .join(df_clean.groupby("Tahun_file").agg(clean=("raw_row", "size"))).fillna(0).astype(int))
ringkas_tahun["excluded"] = ringkas_tahun["raw"] - ringkas_tahun["clean"]
ringkas_tahun["retention_pct"] = (ringkas_tahun["clean"] / ringkas_tahun["raw"] * 100).round(2)
display(ringkas_tahun)

per_kel_tahun = df_clean.groupby(["tahun", "lokasi_key", "kelurahan_master", "kecamatan_master"]).size().rename("n_pengaduan").reset_index()
print("Distribusi jumlah pengaduan per kelurahan-tahun:")
display(per_kel_tahun.groupby("tahun")["n_pengaduan"].describe().round(1))
sedikit = per_kel_tahun[per_kel_tahun["n_pengaduan"] < MIN_N_WARN]
print(f"Kelurahan-tahun dengan < {MIN_N_WARN} pengaduan (embedding rata-rata akan noisy):", len(sedikit))
display(sedikit.head(20))
sedikit.to_csv(OUTPUT_DIR / "kelurahan_tahun_pengaduan_sedikit.csv", index=False)

,raw,clean,excluded,retention_pct
Tahun_file,,,,
2023,12253,12220,33,99.73
2024,11495,11480,15,99.87
2025,10187,10180,7,99.93


Distribusi jumlah pengaduan per kelurahan-tahun:


,count,mean,std,min,25%,50%,75%,max
tahun,,,,,,,,
2023,153.0,79.9,57.8,7.0,40.0,63.0,106.0,366.0
2024,153.0,75.0,54.3,7.0,37.0,57.0,104.0,300.0
2025,153.0,66.5,50.5,4.0,33.0,55.0,85.0,323.0


Kelurahan-tahun dengan < 5 pengaduan (embedding rata-rata akan noisy): 1


,tahun,lokasi_key,kelurahan_master,kecamatan_master,n_pengaduan
440,2025,tambak oso wilangun|benowo,Tambak Oso Wilangun,Benowo,4


## 12. Export


In [ ]:
export = pd.DataFrame({
    "no_tiket": df_clean["No Ticket"],
    "tahun_data": df_clean["Tahun_file"],
    "tahun": df_clean["tahun"].astype(int),
    "tanggal_keluhan": df_clean["tanggal_dt"],
    "instansi": df_clean["Instansi"],
    "topik": df_clean["Topik"],
    "kelompok_topik": df_clean["kelompok_topik"],
    "kategori": df_clean["Kategori"],
    "channel": df_clean["Channel"],
    "teks_keluhan": df_clean["Keluhan_clean"],
    "teks_embedding": df_clean["Keluhan_embedding"],
    "kelurahan": df_clean["kelurahan_master"],
    "kecamatan": df_clean["kecamatan_master"],
    "lokasi_key": df_clean["lokasi_key"],
    "metode_cocok": df_clean["metode_cocok"],
    "lat_kelurahan": df_clean["lat"],
    "lon_kelurahan": df_clean["lon"],
    "sumber_koordinat": df_clean["sumber_koordinat"],
    "kelurahan_raw": df_clean["Kelurahan"],
    "kecamatan_raw": df_clean["Kecamatan"],
    "kecamatan_dikoreksi": df_clean["kecamatan_dikoreksi"],
    "kelurahan_dikoreksi": df_clean["kelurahan_dikoreksi"],
    "keluhan_char_count": df_clean["keluhan_char_count"],
    "keluhan_word_count": df_clean["keluhan_word_count"],
    "flag_dup_teks": df_clean["flag_dup_teks"],
    "flag_telepon": df_clean["flag_telepon"],
    "raw_row": df_clean["raw_row"],
}).reset_index(drop=True)
if "kode_kel" in df_clean.columns:     
    export.insert(export.columns.get_loc("kecamatan") + 1, "kode_kel", df_clean["kode_kel"].to_numpy())

export.to_csv(OUTPUT_DIR / "data_cleaned.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame([
    ("raw_rows", len(df)), ("clean_rows", len(df_clean)), ("excluded_rows", len(excluded)),
    ("excluded_missing_kelurahan_kemunculan", int(alasan["MISSING_KELURAHAN"].sum())),
    ("excluded_missing_keluhan_kemunculan", int(alasan["MISSING_KELUHAN"].sum())),
    ("excluded_invalid_date_kemunculan", int(alasan["INVALID_DATE"].sum())),
    ("excluded_rows_multi_alasan", n_multi),
    ("kecamatan_dikoreksi_rows", int(df_clean["kecamatan_dikoreksi"].sum())),
    ("kelurahan_dikoreksi_rows", int(df_clean["kelurahan_dikoreksi"].sum())),
    ("teks_artefak_backslash_n_dibersihkan", n_artefak_raw),
    ("unique_lokasi_clean", int(df_clean["lokasi_key"].nunique())),
    ("unique_kecamatan_clean", int(df_clean["kecamatan_master"].nunique())),
    ("flag_dup_teks_rows", int(df_clean["flag_dup_teks"].sum())),
    ("flag_telepon_rows", int(df_clean["flag_telepon"].sum())),
    ("kelurahan_tahun_lt_min_n", len(sedikit)),
    ("teks_embedding_dimasking_rows", int((df_clean["Keluhan_embedding"].astype("string") != df_clean["Keluhan_clean"].astype("string")).fillna(False).sum())),   # pada data CLEAN (sama dengan CSV)
    ("teks_embedding_dimasking_rows_raw_sebelum_eksklusi", n_dimasking),                                                                                         # pada seluruh raw (termasuk baris yang dikeluarkan)
] + [(f"cocok_{k}", int(v)) for k, v in df_clean["metode_cocok"].value_counts().items()] + [
], columns=["metric", "value"])
summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)
(OUTPUT_DIR / "cleaning_config.json").write_text(json.dumps({
    "files": {str(k): str(v) for k, v in FILES.items()}, "coord_file": str(COORD_FILE),
    "kelurahan_alias": KELURAHAN_ALIAS, "topik_override": {k if isinstance(k, str) else " | ".join(map(str, k)): v for k, v in TOPIK_OVERRIDE.items()},
    "lokasi_manual": {" | ".join(k): v for k, v in LOKASI_MANUAL.items()}, "mask_telepon": MASK_TELEPON,
    "allow_fallback_coordinate": ALLOW_FALLBACK_COORDINATE,
}, indent=2, ensure_ascii=False), encoding="utf-8")

display(summary)
print("Tersimpan di:", OUTPUT_DIR / "data_cleaned.csv")

,metric,value
0,raw_rows,33935
1,clean_rows,33880
2,excluded_rows,55
3,excluded_missing_kelurahan_kemunculan,51
4,excluded_missing_keluhan_kemunculan,4
5,excluded_invalid_date_kemunculan,0
6,excluded_rows_multi_alasan,0
7,kecamatan_dikoreksi_rows,1
8,kelurahan_dikoreksi_rows,18
9,teks_artefak_backslash_n_dibersihkan,8596


Tersimpan di: c:\Users\DELL\Downloads\Bismillah skripsi lancar jaya nilai A\cleaning_output\data_cleaned.csv
